# Non-SNP-only candidate genes — what does kMate's indel+SV layer flag that SNPs miss?

For every contrast we ran (multitrait **JOINT**, **GLOBAL**, **CLIMATE** × [bio1–19 + PC1 of all
bioclim], and 31 **per-site** scans), we took the clq0.9 LD blocks BH-FDR-significant in the
**non-SNP (indel+SV)** GWAS but **not** in the SNP GWAS — the peaks a SNP-only GWAS would miss —
then mapped each block to overlapping **TAIR10 genes** (within the block span, plus a **±2 kb
promoter flank** to catch nearby regulatory targets) and described each gene via the **Ensembl
Plants** and **UniProt** REST APIs.

**Caveat up front (read before interpreting):** these are, by construction, the *fragile* end of
the signal — blocks one marker class calls significant and the other doesn't. The variance-
partition work showed non-SNP adds ~nothing genome-wide; this list is the scattered local
exceptions. Most are FDR-level (not Bonferroni), and the CLIMATE-contrast subset attaches to
bioclim hits that are likely FDR-tail noise (they mostly vanish under Bonferroni, don't replicate
across classes, and aren't corrected across the 20 collinear climate axes). Treat as a hypothesis-
generating candidate list, not confirmed kMate-unique adaptation loci.

In [1]:

import os
import numpy as np, pandas as pd
os.chdir("/global/scratch/users/tbellg/kmate")
OUT = "results/grenenet_gea/varexp"
g = pd.read_csv(f"{OUT}/nonsnp_only_genes_described.csv").fillna("")
b = pd.read_csv(f"{OUT}/nonsnp_only_blocks.csv").fillna("")
g["desc"] = g["ensembl_description"].str.replace(r" \[Source:.*", "", regex=True)

def ctype(s):
    ks = set()
    for c in s.split(";"):
        if c.startswith("multitrait_CLIMATE"): ks.add("CLIMATE")
        elif c.startswith("multitrait_JOINT"): ks.add("JOINT")
        elif c.startswith("multitrait_GLOBAL"): ks.add("GLOBAL")
        elif c.startswith("persite"): ks.add("per-site")
    return ",".join(sorted(ks))
g["contrast_types"] = g["contrasts"].apply(ctype)
print(f"{len(g)} non-SNP-only candidate genes across {len(b)} blocks")
print(f"  in-block: {(g.overlap_type=='in_block').sum()}   +/-2kb flank-only: {(g.overlap_type=='flank2kb').sum()}")
print(f"  Bonferroni-subset genes: {g.bonferroni.sum()}")
print(f"  with an Ensembl description: {(g.ensembl_description!='').sum()};  with a UniProt function: {(g.uniprot_function!='').sum()}")


126 non-SNP-only candidate genes across 53 blocks
  in-block: 71   +/-2kb flank-only: 55
  Bonferroni-subset genes: 9
  with an Ensembl description: 125;  with a UniProt function: 58


## Bonferroni-stricter subset — the most defensible non-SNP-only hits

Genes under blocks that are non-SNP-only at the **block-level Bonferroni** threshold (not just
FDR) in at least one contrast. This is the subset that survives the strict multiple-testing bar.

In [2]:

bonf = g[g.bonferroni].copy().sort_values(["contrast_types", "gene"])
with pd.option_context("display.max_colwidth", 80, "display.width", 240):
    print(bonf[["gene", "symbol", "desc", "overlap_type", "contrast_types", "contrasts"]].to_string(index=False))


     gene    symbol                                                                                         desc overlap_type   contrast_types                              contrasts
AT4G11270 AT4G11270                                              Transducin/WD40 repeat-like superfamily protein     flank2kb CLIMATE,per-site multitrait_CLIMATE_bio8;persite_site28
AT3G57880 AT3G57880 Calcium-dependent lipid-binding (CaLB domain) plant phosphoribosyltransferase family protein     flank2kb         per-site                         persite_site54
AT3G57890 AT3G57890                                         Tubulin binding cofactor C domain-containing protein     flank2kb         per-site                         persite_site54
AT4G16650 AT4G16650                                                          O-fucosyltransferase family protein     in_block         per-site           persite_site52;persite_site9
AT4G16660 AT4G16660                                                heat shock protein 70 (

## Themed screen — flowering time / cold / heat / circadian

Keyword match over each gene's symbol + Ensembl description + UniProt function text. This is
**high-precision / low-recall**: it catches genes whose annotation *explicitly* names the process,
but a gene with only a generic family name (e.g. "NAC domain protein") won't match even if it is
in fact stress-related. So absence here is not evidence of absence — see the manual notes below.

In [3]:

THEMES = {
    "flowering_time": ["flowering", "floral", "vernaliz", "photoperiod", "inflorescence",
                       "constans", "gigantea", "frigida", "agamous", "apetala", "flc", "soc1",
                       " ft ", "mads", "meristem identity"],
    "cold": ["cold", "freezing", "chilling", "cbf", "dreb", "cor15", "low temperature",
             "ice1", "dehydrin", "cold acclimation", "cold-regulated", "cold regulated"],
    "heat": ["heat shock", "heat stress", "high temperature", "thermotoler", "thermomorph",
             "hsp", "hsf", "chaperone"],
    "circadian": ["circadian", "clock", "cca1", "lhy", "toc1", "pseudo-response regulator",
                  "zeitlupe", "rhythm", " prr", "elf3", "elf4"],
}
text = (g["symbol"].str.lower() + " | " + g["desc"].str.lower() + " | " + g["uniprot_function"].str.lower())
for th, kws in THEMES.items():
    g[th] = text.apply(lambda t: any(k in t for k in kws))
g["themes"] = g.apply(lambda r: ",".join(th for th in THEMES if r[th]), axis=1)
hit = g[g["themes"] != ""]
print(f"{len(hit)} genes matched a theme keyword:")
with pd.option_context("display.max_colwidth", 90, "display.width", 250):
    print(hit[["gene", "symbol", "themes", "desc", "contrast_types"]].to_string(index=False))


8 genes matched a theme keyword:
     gene     symbol                   themes                                            desc contrast_types
AT4G16660  AT4G16660                     heat   heat shock protein 70 (Hsp 70) family protein       per-site
AT1G22770         GI flowering_time,circadian                           gigantea protein (GI)       per-site
AT1G32070        NSI           cold,circadian                     nuclear shuttle interacting        CLIMATE
AT2G31360       ADS2                     cold                         16:0delta9 desaturase 2          JOINT
AT4G12060  AT4G12060                     heat                      Double Clp-N motif protein       per-site
AT4G22670       HIP1                     heat                     HSP70-interacting protein 1        CLIMATE
AT4G28590       MRL7                     heat polyadenylate-binding protein 2-binding protein       per-site
AT5G09350 PI-4KBETA2                     cold          phosphatidylinositol 4-OH kinase beta2  

### UniProt function text for the theme-matched genes (fuller context)

In [4]:

with pd.option_context("display.max_colwidth", 200, "display.width", 260):
    for _, r in hit.iterrows():
        fn = r["uniprot_function"] or "(no UniProt function annotation)"
        print(f"- {r['gene']} ({r['symbol'] or r['desc']}) [{r['themes']}]: {fn}\n")


- AT4G16660 (AT4G16660) [heat]: (no UniProt function annotation)

- AT1G22770 (GI) [flowering_time,circadian]: Involved in regulation of circadian rhythm and photoperiodic flowering. May play a role in maintenance of circadian amplitude and period length. Is involved in phytochrome B signaling. Stabilizes ADO3 and the circadian photoreceptor ADO1/ZTL. Regulates 'CONSTANS' (CO) in the long-day flowering pathway by modulating the ADO3-dependent protein stability of CDF1 and CDF2, but is not essential to activate CO transcription. Regulates, via the microRNA miR172, a CO-independent pathway that promotes photoperiodic flowering by inducing 'FLOWERING LOCUS T'. {ECO:0000269|PubMed:10920210, ECO:0000269|PubMed:17890372, ECO:0000269|PubMed:19619493}.

- AT1G32070 (NSI) [cold,circadian]: Protein acetyltransferase with dual specificity triggering both N-alpha-acetylation (NTA), with a preference for alanine, serine, threonine, methionine and to a lower extent valine as substrates (can also use

## Full annotated table (all 126 genes)

Sorted Bonferroni-first, then by number of contrasts. `overlap_type` = in_block vs ±2kb flank.

In [5]:

show = g.sort_values(["bonferroni", "n_contrasts", "gene"], ascending=[False, False, True])
with pd.option_context("display.max_rows", 200, "display.max_colwidth", 60, "display.width", 260):
    print(show[["gene", "symbol", "desc", "biotype", "overlap_type", "bonferroni",
                "n_contrasts", "contrast_types"]].to_string(index=False))


     gene     symbol                                                                                                        desc        biotype overlap_type  bonferroni  n_contrasts   contrast_types
AT4G11270  AT4G11270                                                             Transducin/WD40 repeat-like superfamily protein protein_coding     flank2kb        True            2 CLIMATE,per-site
AT4G16650  AT4G16650                                                                         O-fucosyltransferase family protein protein_coding     in_block        True            2         per-site
AT4G16660  AT4G16660                                                               heat shock protein 70 (Hsp 70) family protein protein_coding     flank2kb        True            2         per-site
AT3G57880  AT3G57880                Calcium-dependent lipid-binding (CaLB domain) plant phosphoribosyltransferase family protein protein_coding     flank2kb        True            1         per-site
AT3G5

## Notes

- The **JOINT** (any-site selection) genes are the strongest-motivated subset (JOINT is the one
  contrast with a real, well-calibrated genome-wide signal); CLIMATE-contrast genes inherit the
  bioclim-null caveat above.
- Keyword screening is low-recall. A manual pass over the symbols (below the auto-screen) is worth
  doing for canonical stress/flowering/clock genes that carry only generic family annotations.
- Files: `results/grenenet_gea/varexp/nonsnp_only_{blocks,genes,genes_described}.csv`.